#  Day 7: Enterprise Performance Optimization & Maintenance
**Objective:** To implement and benchmark advanced Delta Lake optimization strategies (Liquid Clustering vs. Z-Ordering) and establish data maintenance routines.

### Architecture Highlights:
1. **Dynamic Optimization Setup**: Implementing both Delta 3.0 Liquid Clustering and traditional Partitioning + Z-Order strategies for side-by-side comparison.
2. **Automated Benchmarking Suite**: A custom execution engine to compare latency across high-cardinality, multi-dimensional, and aggregation queries.
3. **Robust Data Mutability (MERGE)**: Handling late-arriving data with resilient multi-format date parsing (`COALESCE(try_to_date...)`).
4. **Data Governance & Maintenance**: Implementing 168-hour `VACUUM` retention policies and creating business-critical churn metrics (`agg_stale_inventory`).

# Environment Setup & Widgets

In [0]:
import time
from pyspark.sql import functions as F

# 1. SETUP WIDGETS FOR DYNAMIC CONFIGURATION
dbutils.widgets.text("project_catalog", "vstone_catalog")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("fact_table_name", "fact_listings")

CATALOG = dbutils.widgets.get("project_catalog")
GOLD = dbutils.widgets.get("gold_schema")
FACT_TABLE = f"{CATALOG}.{GOLD}.{dbutils.widgets.get('fact_table_name')}"

FACT_TABLE_LIQUID = f"{CATALOG}.{GOLD}.fact_listings_liquid"
FACT_TABLE_PARTITIONED = f"{CATALOG}.{GOLD}.fact_listings_partitioned"

print(f" INITIATING PERFORMANCE OPTIMIZATION SUITE FOR: {FACT_TABLE}")
print("="*80)

# Applying Optimization Strategies

In [0]:
# ======================================================================================
# APPROACH 1: LIQUID CLUSTERING (Recommended for Delta 3.0+)
# Cluster Keys: brand, model, location_key, fuel_type
# ======================================================================================
print(" Building Liquid Clustered Table...")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FACT_TABLE_LIQUID}
USING DELTA
CLUSTER BY (brand, model, location_key, fuel_type)
AS SELECT * FROM {FACT_TABLE}
""")

spark.sql(f"OPTIMIZE {FACT_TABLE_LIQUID}")
print(" Liquid Clustering Implemented & Optimized.")

# ======================================================================================
# APPROACH 2: TRADITIONAL PARTITIONING + Z-ORDER
# Partition Key: fuel_type | Z-Order Keys: brand, listing_date
# ======================================================================================
print("\n Building Partitioned & Z-Ordered Table...")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FACT_TABLE_PARTITIONED}
USING DELTA
PARTITIONED BY (fuel_type)
AS SELECT * FROM {FACT_TABLE}
""")

spark.sql(f"OPTIMIZE {FACT_TABLE_PARTITIONED} ZORDER BY (brand, listing_date)")
print(" Partitioning + Z-Order Implemented.")
print("="*80)

# Automated Benchmarking Framework

In [0]:
# ======================================================================================
# BENCHMARKING ENGINE
# ======================================================================================
def benchmark(query: str, label: str, runs: int = 3):
    """Executes a query multiple times to calculate average latency."""
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        spark.sql(query).collect()
        elapsed = round(time.perf_counter() - start, 3)
        times.append(elapsed)
    avg_t = round(sum(times) / len(times), 3)
    print(f"   [{label}] Avg Execution: {avg_t}s")
    return avg_t

# Define Test Queries
queries = [
    ("Q1: Brand Filter (VOLVO)", 
     f"SELECT * FROM {FACT_TABLE_LIQUID} WHERE brand = 'VOLVO'", 
     f"SELECT * FROM {FACT_TABLE_PARTITIONED} WHERE brand = 'VOLVO'"),

    ("Q2: Monthly Brand Count", 
     f"SELECT brand, COUNT(*) FROM {FACT_TABLE_LIQUID} GROUP BY brand", 
     f"SELECT brand, COUNT(*) FROM {FACT_TABLE_PARTITIONED} GROUP BY brand"),

    ("Q3: City & Price Category", 
     f"SELECT location_key, AVG(price_rub) as avg_p FROM {FACT_TABLE_LIQUID} WHERE location_key = 'Moscow' AND price_category = 'LUXURY' GROUP BY location_key", 
     f"SELECT location_key, AVG(price_rub) as avg_p FROM {FACT_TABLE_PARTITIONED} WHERE location_key = 'Moscow' AND price_category = 'LUXURY' GROUP BY location_key"),

    ("Q4: Multi-Brand Price Analysis", 
     f"SELECT brand, model, COUNT(*) FROM {FACT_TABLE_LIQUID} WHERE brand IN ('TOYOTA', 'NISSAN', 'KIA') AND mileage_km < 50000 GROUP BY brand, model", 
     f"SELECT brand, model, COUNT(*) FROM {FACT_TABLE_PARTITIONED} WHERE brand IN ('TOYOTA', 'NISSAN', 'KIA') AND mileage_km < 50000 GROUP BY brand, model")
]

# Execute & Log Results
benchmark_results = []
print(f" STARTING BENCHMARK EXECUTION ({len(queries)} Queries)...")

for label, ql, qp in queries:
    print(f"\n--- Testing {label} ---")
    t_liq = benchmark(ql, "Liquid Clustering")
    t_zord = benchmark(qp, "Partitioned + Z-Order")
    
    winner = "Liquid" if t_liq <= t_zord else "Z-Order"
    speedup = round(max(t_liq, t_zord) / min(t_liq, t_zord), 2)
    print(f" Winner: {winner} ({speedup}x faster)")
    
    benchmark_results.append((label, float(t_liq), float(t_zord), winner))

# Save Audit Log
results_schema = "query STRING, liquid_avg_secs DOUBLE, zorder_avg_secs DOUBLE, winner STRING"
spark.createDataFrame(benchmark_results, results_schema) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD}.benchmark_results")

print("\n Benchmark results documented in Gold Layer.")

# Advanced Mutability (Merge Into logic)

In [0]:
# ======================================================================================
# DATA MUTABILITY: Handling Late-Arriving Data with MERGE
# ======================================================================================
print(" RUNNING ROBUST MERGE INTO ALGORITHM...")

spark.sql(f"""
MERGE INTO {FACT_TABLE_LIQUID} AS target
USING (
    SELECT
        *,
        -- Multi-format date handler
        COALESCE(
            try_to_date(cast(listing_date as string), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
            try_to_date(cast(listing_date as string), 'dd.MM.yyyy')
        ) as corrected_date,
        current_timestamp() as new_load_dt
    FROM {FACT_TABLE} LIMIT 500
) AS source
ON target.listing_id = source.listing_id
WHEN MATCHED AND target.price_rub != source.price_rub THEN
    UPDATE SET
        target.price_rub = source.price_rub,
        target.price_usd = source.price_usd,
        target.gold_load_dt = source.new_load_dt
WHEN NOT MATCHED THEN
    INSERT (
        listing_id, brand, model, year, listing_date, price_rub, price_usd, price_category, 
        fuel_type, transmission_type, engine_power, mileage_km, car_age_at_listing, 
        is_high_mileage, price_per_hp_usd, location_key, gold_load_dt
    )
    VALUES (
        source.listing_id, source.brand, source.model, source.year, source.corrected_date,
        source.price_rub, source.price_usd, source.price_category, source.fuel_type,
        source.transmission_type, source.engine_power, source.mileage_km,
        source.car_age_at_listing, source.is_high_mileage, source.price_per_hp_usd,
        source.location_key, source.new_load_dt
    )
""")
print(" Merge operation complete.")

# Governance, Maintenance & Churn Metrics

In [0]:
# ======================================================================================
# DATA GOVERNANCE & BUSINESS KPI CREATION
# ======================================================================================

# 1. Delta Maintenance (VACUUM)
print("\n RUNNING DELTA MAINTENANCE (168-Hour Retention)...")
for table in ["fact_listings_liquid", "fact_listings_partitioned"]:
    spark.sql(f"VACUUM {CATALOG}.{GOLD}.{table} RETAIN 168 HOURS")

# 2. Business KPI: Churn Metric Generation
print("\n GENERATING CHURN METRICS (Stale Inventory > 180 Days)...")
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD}.agg_stale_inventory AS
WITH normalized_dates AS (
    SELECT brand, model, location_key,
        COALESCE(
            try_to_date(cast(listing_date as string), "yyyy-MM-dd'T'HH:mm:ss'Z'"), 
            try_to_date(cast(listing_date as string), 'dd.MM.yyyy')
        ) as clean_date
    FROM {FACT_TABLE_LIQUID}
)
SELECT brand, model, location_key, MAX(clean_date) as last_seen_date,
       DATEDIFF(current_date(), MAX(clean_date)) as days_inactive
FROM normalized_dates
GROUP BY brand, model, location_key
HAVING days_inactive > 180
""")

# 3. Enterprise Metadata Tagging
spark.sql(f"COMMENT ON TABLE {FACT_TABLE_LIQUID} IS 'Primary optimized fact table using Liquid Clustering for high-cardinality filters.'")
spark.sql(f"COMMENT ON TABLE {CATALOG}.{GOLD}.agg_stale_inventory IS 'Churn Metric: Car models with no listing activity for over 180 days.'")

# 4. Final Verification Displays
print("\n OPTIMIZATION STRATEGY COMPARISON RESULTS:")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD}.benchmark_results ORDER BY winner DESC"))

print("\n TOP 10 STALE INVENTORY (CHURN):")
display(spark.table(f"{CATALOG}.{GOLD}.agg_stale_inventory").orderBy(F.desc("days_inactive")).limit(10))


In [0]:
import time
from pyspark.sql import functions as F
# Project Standard Naming

CATALOG = "vstone_catalog"
GOLD = "gold"
FACT_TABLE = f"{CATALOG}.{GOLD}.fact_listings"
print(f" Starting Performance Optimization for {FACT_TABLE}...")

# ======================================================================================
# APPROACH 1: LIQUID CLUSTERING (Recommended for Delta 3.0+)
# ======================================================================================

# Using location_key and fuel_type as per our project schema

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD}.fact_listings_liquid
USING DELTA
CLUSTER BY (brand, model, location_key, fuel_type)
AS SELECT * FROM {FACT_TABLE}
""")

spark.sql(f"OPTIMIZE {CATALOG}.{GOLD}.fact_listings_liquid")
print(" Liquid Clustering implemented.")

# ======================================================================================
# APPROACH 2: TRADITIONAL PARTITIONING + Z-ORDER
# ======================================================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD}.fact_listings_partitioned
USING DELTA
PARTITIONED BY (fuel_type)
AS SELECT * FROM {FACT_TABLE}
""")

spark.sql(f"""
OPTIMIZE {CATALOG}.{GOLD}.fact_listings_partitioned
ZORDER BY (brand, listing_date)
""")
print(" Partitioning + Z-Order implemented.")

# ======================================================================================
# BENCHMARKING FUNCTION (Serverless Compatible)
# ======================================================================================

def benchmark(query: str, label: str, runs: int = 3):
    times = []
    for _ in range(runs):
        # spark.catalog.clearCache() is skipped for Serverless compatibility
        start = time.perf_counter()
        spark.sql(query).collect()
        elapsed = round(time.perf_counter() - start, 3)
        times.append(elapsed)
    avg_t = round(sum(times) / len(times), 3)
    print(f"   [{label}] Avg Execution: {avg_t}s")
    return avg_t

# ======================================================================================
# EXECUTION & RESULTS (Schema Fixed)
# ======================================================================================

# ======================================================================================
# EXPANDED BENCHMARK TEST QUERIES (Day 7 Requirement)
# ======================================================================================
queries = [
    # Q1: High Cardinality Filter (Standard BI Search)
    ("Q1: Brand Filter (VOLVO)", 
     f"SELECT * FROM {CATALOG}.{GOLD}.fact_listings_liquid WHERE brand = 'VOLVO'", 
     f"SELECT * FROM {CATALOG}.{GOLD}.fact_listings_partitioned WHERE brand = 'VOLVO'"),

    # Q2: Analytical Aggregation (Global Trend)
    ("Q2: Monthly Brand Count", 
     f"SELECT brand, COUNT(*) FROM {CATALOG}.{GOLD}.fact_listings_liquid GROUP BY brand", 
     f"SELECT brand, COUNT(*) FROM {CATALOG}.{GOLD}.fact_listings_partitioned GROUP BY brand"),

    # Q3: Multi-Dimension Filter (Testing Co-locality)
    ("Q3: City & Price Category", 
     f"""SELECT location_key, AVG(price_rub) as avg_p 
         FROM {CATALOG}.{GOLD}.fact_listings_liquid 
         WHERE location_key = 'Moscow' AND price_category = 'LUXURY' 
         GROUP BY location_key""", 
     f"""SELECT location_key, AVG(price_rub) as avg_p 
         FROM {CATALOG}.{GOLD}.fact_listings_partitioned 
         WHERE location_key = 'Moscow' AND price_category = 'LUXURY' 
         GROUP BY location_key"""),

    # Q4: Complex In-List & Range (Comparative Performance)
    ("Q4: Multi-Brand Price Analysis", 
     f"""SELECT brand, model, COUNT(*) 
         FROM {CATALOG}.{GOLD}.fact_listings_liquid 
         WHERE brand IN ('TOYOTA', 'NISSAN', 'KIA') AND mileage_km < 50000 
         GROUP BY brand, model""", 
     f"""SELECT brand, model, COUNT(*) 
         FROM {CATALOG}.{GOLD}.fact_listings_partitioned 
         WHERE brand IN ('TOYOTA', 'NISSAN', 'KIA') AND mileage_km < 50000 
         GROUP BY brand, model""")
]

# ======================================================================================
# EXECUTION & RESULTS LOGGING
# ======================================================================================
benchmark_results = []
print(f"\n📊 Starting Benchmarking for {len(queries)} Queries...")

for label, ql, qp in queries:
    print(f"\n--- Testing {label} ---")
    
    # Executing on Liquid Table
    t_liq = benchmark(ql, "Liquid Clustering")
    
    # Executing on Partitioned/Z-Order Table
    t_zord = benchmark(qp, "Partitioned + Z-Order")
    
    # Determine Winner
    winner = "Liquid" if t_liq <= t_zord else "Z-Order"
    
    # Speedup calculation
    speedup = round(max(t_liq, t_zord) / min(t_liq, t_zord), 2)
    print(f"🏆 Winner: {winner} ({speedup}x faster)")
    
    benchmark_results.append((label, float(t_liq), float(t_zord), winner))

# Documentation: Saving results for project audit
results_schema = "query STRING, liquid_avg_secs DOUBLE, zorder_avg_secs DOUBLE, winner STRING"
spark.createDataFrame(benchmark_results, results_schema) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD}.benchmark_results")

print("\n✓ Benchmark results successfully documented in the Gold layer.")

# Fix: overwriteSchema handled for existing table
results_schema = "query STRING, liquid_avg_secs DOUBLE, zorder_avg_secs DOUBLE, winner STRING"
spark.createDataFrame(benchmark_results, results_schema) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD}.benchmark_results")

# ======================================================================================
# FIXED MERGE INTO: Handling Multi-format Dates & Ambiguity
# ======================================================================================

print("\n Running FIXED MERGE INTO for late-arriving data...")

spark.sql(f"""
MERGE INTO {CATALOG}.{GOLD}.fact_listings_liquid AS target
USING (
    SELECT
        *,
        -- Using COALESCE and try_to_date to handle both ISO and dd.MM.yyyy formats
        COALESCE(
            try_to_date(cast(listing_date as string), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
            try_to_date(cast(listing_date as string), 'dd.MM.yyyy')
        ) as corrected_date,
        current_timestamp() as new_load_dt
    FROM {FACT_TABLE} LIMIT 500
) AS source
ON target.listing_id = source.listing_id
WHEN MATCHED AND target.price_rub != source.price_rub THEN
    UPDATE SET
        target.price_rub = source.price_rub,
        target.price_usd = source.price_usd,
        target.gold_load_dt = source.new_load_dt
WHEN NOT MATCHED THEN
    INSERT (
        listing_id, brand, model, year, listing_date,
        price_rub, price_usd, price_category, fuel_type,
        transmission_type, engine_power, mileage_km,
        car_age_at_listing, is_high_mileage, price_per_hp_usd,
        location_key, gold_load_dt
    )
    VALUES (
        source.listing_id, source.brand, source.model, source.year, source.corrected_date,
        source.price_rub, source.price_usd, source.price_category, source.fuel_type,
        source.transmission_type, source.engine_power, source.mileage_km,
        source.car_age_at_listing, source.is_high_mileage, source.price_per_hp_usd,
        source.location_key, source.new_load_dt
    )
""")


In [0]:
import time
from pyspark.sql import functions as F

# Project Standard Naming
CATALOG = "vstone_catalog"
GOLD = "gold"
FACT_TABLE_LIQUID = f"{CATALOG}.{GOLD}.fact_listings_liquid"

# ======================================================================================
# 1. DELTA MAINTENANCE (VACUUM & HISTORY)
# ======================================================================================
print("\n=== 🧹 RUNNING DELTA MAINTENANCE ===")
for table in ["fact_listings_liquid", "fact_listings_partitioned"]:
    # Cleanup old snapshots older than 7 days
    spark.sql(f"VACUUM {CATALOG}.{GOLD}.{table} RETAIN 168 HOURS")
    print(f"✓ VACUUM complete: {table}")

print("\n=== 📜 DELTA HISTORY (Liquid Table) ===")
display(spark.sql(f"DESCRIBE HISTORY {FACT_TABLE_LIQUID}").limit(5))

# ======================================================================================
# 2. BUSINESS KPI: CHURN METRIC (Multi-Format Date Fix)
# ======================================================================================
print("\n🚀 Building Churn Metric (Stale Listings)...")

# FIX: Using COALESCE to handle both ISO and dd.MM.yyyy formats
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD}.agg_stale_inventory AS
WITH normalized_dates AS (
    SELECT 
        brand, model, location_key,
        COALESCE(
            try_to_date(cast(listing_date as string), "yyyy-MM-dd'T'HH:mm:ss'Z'"), 
            try_to_date(cast(listing_date as string), 'dd.MM.yyyy')
        ) as clean_date
    FROM {FACT_TABLE_LIQUID}
)
SELECT 
    brand, 
    model, 
    location_key, 
    MAX(clean_date) as last_seen_date,
    DATEDIFF(current_date(), MAX(clean_date)) as days_inactive
FROM normalized_dates
GROUP BY brand, model, location_key
HAVING days_inactive > 180
""")

print("📊 Churn Metric Results (Top 10 Stale Inventory):")
display(spark.table(f"{CATALOG}.{GOLD}.agg_stale_inventory").orderBy(F.desc("days_inactive")).limit(10))

# ======================================================================================
# 3. DISCOVERABILITY: ENTERPRISE METADATA
# ======================================================================================
print("🚀 Adding Enterprise Metadata...")
spark.sql(f"COMMENT ON TABLE {FACT_TABLE_LIQUID} IS 'Primary optimized fact table using Liquid Clustering for high-cardinality filters.'")
spark.sql(f"COMMENT ON TABLE {CATALOG}.{GOLD}.agg_stale_inventory IS 'Churn Metric: Car models with no listing activity for over 180 days.'")

# ======================================================================================
# 4. FINAL PERFORMANCE SUMMARY
# ======================================================================================
print("\n=== 🏆 OPTIMIZATION STRATEGY COMPARISON ===")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD}.benchmark_results ORDER BY winner DESC"))

print("\n🎯 Day 7: Performance, Churn, and Documentation Complete!")